### Bài 1: Xây dựng mô hình LeNet. Huấn luyện và đánh giá mô hình LeNet trên 4 độ đo precision, recall và F1-macro (sử dụng Adam làm optimizer).

![LeNet](../images/01.png)

In [1]:
# ==============================================================================
# CÀI ĐẶT CHUNG & HÀM TIỆN ÍCH (TỪ PHẦN A & PHẦN B.7 TRONG TÀI LIỆU)
# ==============================================================================

import time
import math
from pathlib import Path
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import DataLoader
from torchvision import datasets, transforms
from sklearn.metrics import classification_report, confusion_matrix, accuracy_score, precision_score, recall_score, f1_score
import matplotlib.pyplot as plt
import seaborn as sns



# Cố định seed giống hướng dẫn
SEED = 42
torch.manual_seed(SEED)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("torch  :", torch.__version__)
print("device :", device)



# Đường dẫn dữ liệu theo cấu trúc thư mục của bạn
DATA_ROOT = Path("VinaFood21")
BATCH = 32
EPOCHS = 10  # Bạn có thể tăng lên 20 nếu máy có GPU mạnh
MEAN, STD = [0.485, 0.456, 0.406], [0.229, 0.224, 0.225]



# Hai hàm tiện ích dùng lại trong cả bài (Phần A.1)
def n_params(module, trainable_only=False):
    ps = module.parameters()
    if trainable_only:
        ps = (p for p in module.parameters() if p.requires_grad)
    return sum(p.numel() for p in ps)



def show(label, t):
    print(f"{label:<28} {tuple(t.shape)}")



# Hàm tạo DataLoader theo kích thước ảnh (Phần A.2)
def get_dataloaders(img_size, use_augment=True):
    if use_augment:
        train_tf = transforms.Compose([
            transforms.RandomResizedCrop(img_size, scale=(0.7, 1.0)),
            transforms.RandomHorizontalFlip(),
            transforms.RandomRotation(15),
            transforms.ColorJitter(brightness=0.2, contrast=0.2, saturation=0.2),
            transforms.ToTensor(),
            transforms.Normalize(MEAN, STD),
        ])
    else:
        train_tf = transforms.Compose([
            transforms.Resize((img_size, img_size)),
            transforms.ToTensor(),
            transforms.Normalize(MEAN, STD),
        ])



    eval_tf = transforms.Compose([
        transforms.Resize((img_size, img_size)),
        transforms.ToTensor(),
        transforms.Normalize(MEAN, STD),
    ])

    train_ds = datasets.ImageFolder(DATA_ROOT / "train", transform=train_tf)
    val_ds   = datasets.ImageFolder(DATA_ROOT / "val", transform=eval_tf)
    test_ds  = datasets.ImageFolder(DATA_ROOT / "test", transform=eval_tf)



    train_dl = DataLoader(train_ds, batch_size=BATCH, shuffle=True, num_workers=2)
    val_dl   = DataLoader(val_ds, batch_size=BATCH, shuffle=False, num_workers=2)
    test_dl  = DataLoader(test_ds, batch_size=BATCH, shuffle=False, num_workers=2)
    return train_dl, val_dl, test_dl, train_ds.classes



# Hai hàm huấn luyện và đánh giá dùng chung (Phần B.7 Bước 1)

def train_one_epoch(model, loader, criterion, optimizer, device):
    model.train()
    total, correct, running = 0, 0, 0.0
    for xb, yb in loader:
        xb, yb = xb.to(device), yb.to(device)
        optimizer.zero_grad()
        logits = model(xb)
        loss = criterion(logits, yb)
        loss.backward()
        optimizer.step()
        running += loss.item() * yb.size(0)
        correct += (logits.argmax(1) == yb).sum().item()
        total += yb.size(0)
    return running / total, correct / total



@torch.no_grad()
def evaluate(model, loader, criterion, device):
    model.eval()
    total, correct, running = 0, 0, 0.0
    y_true, y_pred = [], []
    for xb, yb in loader:
        xb, yb = xb.to(device), yb.to(device)
        logits = model(xb)
        running += criterion(logits, yb).item() * yb.size(0)
        pred = logits.argmax(1)
        correct += (pred == yb).sum().item()
        total += yb.size(0)
        y_true += yb.cpu().tolist()
        y_pred += pred.cpu().tolist()
    return running / total, correct / total, y_true, y_pred



# Hàm chạy vòng lặp huấn luyện, vẽ đồ thị và in 4 độ đo (Phần B.7 Bước 2 & 3)

def run_experiment(model, train_dl, val_dl, test_dl, classes, lr=1e-3, epochs=EPOCHS, title="Model"):
    print(f"\n==================== {title} ====================")
    print("Tổng tham số         :", f"{n_params(model):,}")
    print("Tham số được cập nhật:", f"{n_params(model, trainable_only=True):,}")
    model = model.to(device)
    criterion = nn.CrossEntropyLoss()
    optimizer = torch.optim.Adam(filter(lambda p: p.requires_grad, model.parameters()), lr=lr)
    lich_su = []
    start_time = time.time()
    for ep in range(1, epochs + 1):
        tr_loss, tr_acc = train_one_epoch(model, train_dl, criterion, optimizer, device)
        va_loss, va_acc, _, _ = evaluate(model, val_dl, criterion, device)
        lich_su.append((ep, tr_loss, va_loss))
        print(f"epoch {ep:>2}  train {tr_loss:.4f}/{tr_acc:.4f}   val {va_loss:.4f}/{va_acc:.4f}")

    sec_per_epoch = (time.time() - start_time) / epochs
    print(f"Thời gian trung bình : {sec_per_epoch:.2f} giây/epoch")
    # Vẽ đường loss train và val
    ep, tr, va = zip(*lich_su)
    plt.figure(figsize=(6, 4))
    plt.plot(ep, tr, label="train loss")
    plt.plot(ep, va, label="val loss")
    plt.title(f"Loss - {title}")
    plt.xlabel("epoch")
    plt.ylabel("loss")
    plt.legend()
    plt.show()



    # Đánh giá trên tập test với 4 độ đo
    _, _, y_true, y_pred = evaluate(model, test_dl, criterion, device)
    acc  = accuracy_score(y_true, y_pred)
    prec = precision_score(y_true, y_pred, average="macro", zero_division=0)
    rec  = recall_score(y_true, y_pred, average="macro", zero_division=0)
    f1   = f1_score(y_true, y_pred, average="macro", zero_division=0)
    print(f"\n--- Kết quả trên tập Test ({title}) ---")
    print(f"Accuracy  : {acc:.4f}")
    print(f"Precision : {prec:.4f}")
    print(f"Recall    : {rec:.4f}")
    print(f"Macro-F1  : {f1:.4f}\n")
    print("Bảng F1 từng lớp:")
    print(classification_report(y_true, y_pred, target_names=classes, digits=4, zero_division=0))



    # Vẽ Confusion Matrix 21x21
    cm = confusion_matrix(y_true, y_pred)
    plt.figure(figsize=(10, 8))
    sns.heatmap(cm, xticklabels=classes, yticklabels=classes, cmap="Blues", annot=False)
    plt.title(f"Confusion Matrix - {title}")
    plt.xlabel("Predicted")
    plt.ylabel("True")
    plt.show()
    return f1

torch  : 2.14.1+cpu
device : cpu


In [2]:
# ==============================================================================

# BÀI 1: XÂY DỰNG MÔ HÌNH LENET-5 (TỪ PHẦN B.3 & C.1 BÀI 1)

# ==============================================================================

# Định nghĩa LeNet-5 với số channel vào, kích thước ảnh và số lớp tham số hoá

class LeNet5(nn.Module):
    def __init__(self, in_ch=3, img=64, num_classes=21, pool="avg"):
        super().__init__()
        Pool = nn.AvgPool2d if pool == "avg" else nn.MaxPool2d
        self.c1 = nn.Conv2d(in_ch, 6, kernel_size=5)
        self.s2 = Pool(2, 2)
        self.c3 = nn.Conv2d(6, 16, kernel_size=5)
        self.s4 = Pool(2, 2)
        n_flat = self._flat_dim(in_ch, img)
        self.f5 = nn.Linear(n_flat, 120)
        self.f6 = nn.Linear(120, 84)
        self.out = nn.Linear(84, num_classes)
        self.n_flat = n_flat

    def _flat_dim(self, in_ch, img):
        with torch.no_grad():
            z = torch.zeros(1, in_ch, img, img)
            z = self.s4(self.c3(self.s2(self.c1(z))))
        return int(z.flatten(1).shape[1])



    def forward(self, x):
        x = self.s2(torch.relu(self.c1(x)))
        x = self.s4(torch.relu(self.c3(x)))
        x = x.flatten(1)
        x = torch.relu(self.f5(x))
        x = torch.relu(self.f6(x))
        return self.out(x)



# Hàm in bảng tham số từng layer của LeNet-5 (Phần B.3 Bước 2)

def layer_table(model, img, in_ch=3):

    x = torch.zeros(1, in_ch, img, img)

    rows, total = [], 0

    for name, layer in [("C1", model.c1), ("S2", model.s2),

                        ("C3", model.c3), ("S4", model.s4)]:

        x = layer(x)

        p = n_params(layer)

        total += p

        rows.append((name, "x".join(map(str, tuple(x.shape)[1:])), p))

    x = x.flatten(1)

    rows.append(("flatten", str(x.shape[1]), 0))

    for name, layer in [("F5", model.f5), ("F6", model.f6), ("OUT", model.out)]:

        x = layer(x)

        p = n_params(layer)

        total += p

        rows.append((name, str(x.shape[1]), p))

    return rows, total



# --- Chạy Bài 1 ---

train_dl_64, val_dl_64, test_dl_64, CLASSES = get_dataloaders(img_size=64, use_augment=True)



lenet_avg = LeNet5(in_ch=3, img=64, num_classes=len(CLASSES), pool="avg")

rows, total = layer_table(lenet_avg, 64)

print(f"{'layer':<10}{'output':<16}{'tham số':>12}")

for name, shape, p in rows:

    print(f"{name:<10}{shape:<16}{p:>12,}")

print(f"{'TỔNG':<10}{'':<16}{total:>12,}")

# Huấn luyện bản AvgPool và MaxPool để so sánh theo yêu cầu đề bài
f1_avg = run_experiment(lenet_avg, train_dl_64, val_dl_64, test_dl_64, CLASSES, lr=1e-3, title="Bài 1: LeNet-5 (AvgPool)")
lenet_max = LeNet5(in_ch=3, img=64, num_classes=len(CLASSES), pool="max")
f1_max = run_experiment(lenet_max, train_dl_64, val_dl_64, test_dl_64, CLASSES, lr=1e-3, title="Bài 1: LeNet-5 (MaxPool)")
print(f"So sánh Bài 1 -> Macro-F1 AvgPool: {f1_avg:.4f} | Macro-F1 MaxPool: {f1_max:.4f}")

FileNotFoundError: Couldn't find any class folder in VinaFood21\train.

### Bài 2: Xây dựng mô hình GoogLeNet. Huấn luyện và đánh giá mô hình GoogLeNet trên 4 độ đo precision, recall và F1 (sử dụng Adam làm optimizer).

**Inception Blocks**

![Inception Blocks](../images/02.png)

**GoogLeNet (Multi-branch Network)**

![GoogLeNe](../images/03.png)

In [ ]:
# ==============================================================================

# BÀI 2: XÂY DỰNG MÔ HÌNH GOOGLENET RÚT GỌN (TỪ PHẦN B.4 & SLIDE A.6)

# ==============================================================================

# Định nghĩa Inception block (Phần B.4 Bước 1)
class InceptionBlock(nn.Module):
    def __init__(self, in_ch, f1, f3r, f3, f5r, f5, fp):
        super().__init__()

        self.b1 = nn.Sequential(

            nn.Conv2d(in_ch, f1, 1), nn.ReLU())

        self.b2 = nn.Sequential(

            nn.Conv2d(in_ch, f3r, 1), nn.ReLU(),

            nn.Conv2d(f3r, f3, 3, padding=1), nn.ReLU())

        self.b3 = nn.Sequential(

            nn.Conv2d(in_ch, f5r, 1), nn.ReLU(),

            nn.Conv2d(f5r, f5, 5, padding=2), nn.ReLU())

        self.b4 = nn.Sequential(

            nn.MaxPool2d(3, stride=1, padding=1),

            nn.Conv2d(in_ch, fp, 1), nn.ReLU())



    def forward(self, x):

        # dim=1 là chiều channel

        return torch.cat([self.b1(x), self.b2(x),

                          self.b3(x), self.b4(x)], dim=1)



# Định nghĩa GoogLeNet rút gọn (3 Inception block + Global Average Pooling)

class MiniGoogLeNet(nn.Module):

    def __init__(self, in_ch=3, num_classes=21):

        super().__init__()

        self.stem = nn.Sequential(

            nn.Conv2d(in_ch, 32, kernel_size=3, stride=1, padding=1),

            nn.ReLU(),

            nn.MaxPool2d(2, 2)  # 64x64 -> 32x32

        )

        # Block 1: ra 16 + 24 + 8 + 8 = 56 channel

        self.inc1 = InceptionBlock(32, 16, 16, 24, 4, 8, 8)

        self.pool1 = nn.MaxPool2d(2, 2)  # 32x32 -> 16x16



        # Block 2: ra 32 + 48 + 16 + 16 = 112 channel (y hệt B.4 Bước 3)

        self.inc2 = InceptionBlock(56, 32, 32, 48, 8, 16, 16)

        self.pool2 = nn.MaxPool2d(2, 2)  # 16x16 -> 8x8



        # Block 3: ra 64 + 96 + 32 + 32 = 224 channel

        self.inc3 = InceptionBlock(112, 64, 48, 96, 16, 32, 32)



        # Dùng AdaptiveAvgPool2d(1) thay cho Flatten() theo yêu cầu Bài 2

        self.gap = nn.AdaptiveAvgPool2d(1)

        self.fc = nn.Linear(224, num_classes)



    def forward(self, x):

        x = self.stem(x)

        x = self.pool1(self.inc1(x))

        x = self.pool2(self.inc2(x))

        x = self.inc3(x)

        x = self.gap(x).flatten(1)

        return self.fc(x)



# --- Chạy Bài 2 ---

googlenet = MiniGoogLeNet(in_ch=3, num_classes=len(CLASSES))



# In kiểm tra số channel sau mỗi Inception block trước khi huấn luyện

x_test = torch.rand(1, 3, 64, 64)

with torch.no_grad():

    s = googlenet.stem(x_test)

    b1_out = googlenet.inc1(s)

    b2_out = googlenet.inc2(googlenet.pool1(b1_out))

    b3_out = googlenet.inc3(googlenet.pool2(b2_out))

    show("Sau stem", s)

    show("Sau Inception 1 (56 ch)", b1_out)

    show("Sau Inception 2 (112 ch)", b2_out)

    show("Sau Inception 3 (224 ch)", b3_out)



f1_googlenet = run_experiment(googlenet, train_dl_64, val_dl_64, test_dl_64, CLASSES, lr=1e-3, title="Bài 2: GoogLeNet rút gọn")


### Bài 3*: Xây dựng mô hình ResNet-18, đánh giá mô hình ResNet-18 trên bộ dữ liệu VinaFood21 sử dụng các độ đo precision, recall, và F1 (Sử dụng Adam làm optimizer).

**ResNet Block**

![ResNet Block](../images/04.png)

**ResNet**

![ResNet](../images/05.png)

In [ ]:
# ==============================================================================

# BÀI 3*: XÂY DỰNG MÔ HÌNH RESNET-18 TỪ ĐẦU (TỪ PHẦN B.5 & SLIDE A.7)

# ==============================================================================

# Định nghĩa BasicBlock (Phần B.5 Bước 1)

class BasicBlock(nn.Module):

    def __init__(self, in_ch, out_ch, stride=1):

        super().__init__()

        self.conv1 = nn.Conv2d(in_ch, out_ch, 3, stride, padding=1, bias=False)

        self.bn1 = nn.BatchNorm2d(out_ch)

        self.conv2 = nn.Conv2d(out_ch, out_ch, 3, 1, padding=1, bias=False)

        self.bn2 = nn.BatchNorm2d(out_ch)



        self.shortcut = nn.Identity()

        if stride != 1 or in_ch != out_ch:       # cần projection

            self.shortcut = nn.Sequential(

                nn.Conv2d(in_ch, out_ch, 1, stride, bias=False),

                nn.BatchNorm2d(out_ch))



    def forward(self, x):

        out = torch.relu(self.bn1(self.conv1(x)))

        out = self.bn2(self.conv2(out))

        out = out + self.shortcut(x)             # cộng trước

        return torch.relu(out)                   # ReLU sau



# Định nghĩa ResNet-18 theo bốn stage [2, 2, 2, 2] (Phần B.5 Bước 3)

class ResNet18(nn.Module):

    def __init__(self, num_classes=21):

        super().__init__()

        self.stem = nn.Sequential(

            nn.Conv2d(3, 64, 7, stride=2, padding=3, bias=False),

            nn.BatchNorm2d(64), nn.ReLU(),

            nn.MaxPool2d(3, stride=2, padding=1))

        self.layer1 = self._stage(64, 64, stride=1)

        self.layer2 = self._stage(64, 128, stride=2)

        self.layer3 = self._stage(128, 256, stride=2)

        self.layer4 = self._stage(256, 512, stride=2)

        self.pool = nn.AdaptiveAvgPool2d(1)

        self.fc = nn.Linear(512, num_classes)



    @staticmethod

    def _stage(in_ch, out_ch, stride):

        return nn.Sequential(BasicBlock(in_ch, out_ch, stride),

                             BasicBlock(out_ch, out_ch, 1))



    def forward(self, x):

        x = self.stem(x)

        for layer in (self.layer1, self.layer2, self.layer3, self.layer4):

            x = layer(x)

        return self.fc(self.pool(x).flatten(1))



# --- Chạy Bài 3* (Resize 96x96 theo hướng dẫn B.5 Bước 4, chạy 2 lần: có và không có Augmentation) ---

# Lần 1: Không có Augmentation

train_dl_96_noaug, val_dl_96, test_dl_96, _ = get_dataloaders(img_size=96, use_augment=False)

r18_no_aug = ResNet18(num_classes=len(CLASSES))

f1_r18_noaug = run_experiment(r18_no_aug, train_dl_96_noaug, val_dl_96, test_dl_96, CLASSES, lr=1e-3, title="Bài 3*: ResNet-18 (Không Augmentation)")



# Lần 2: Có Augmentation

train_dl_96_aug, _, _, _ = get_dataloaders(img_size=96, use_augment=True)

r18_aug = ResNet18(num_classes=len(CLASSES))

f1_r18_aug = run_experiment(r18_aug, train_dl_96_aug, val_dl_96, test_dl_96, CLASSES, lr=1e-3, title="Bài 3*: ResNet-18 (Có Augmentation)")



print(f"So sánh Bài 3* -> Macro-F1 Không Augment: {f1_r18_noaug:.4f} | Có Augment: {f1_r18_aug:.4f}")

### Bài 4*: Sử dụng pretrained ResNet50 từ HuggingFace để fine-tune trên bộ dữ liệu VinaFood21.

In [ ]:
# ==============================================================================

# BÀI 4*: FINE-TUNE RESNET-50 PRETRAINED (TỪ PHẦN B.6 & SLIDE A.8)

# ==============================================================================

from transformer/s import ResNetForImageClassification
# Định nghĩa model: backbone pretrained + classifier mới 21 lớp (Phần B.6 Bước 2)

class PretrainedResnet(nn.Module):

    def __init__(self, num_classes=21, freeze_backbone=False):

        super().__init__()

        base = ResNetForImageClassification.from_pretrained("microsoft/resnet-50")

        self.resnet = base.resnet

        self.classifier = nn.Linear(2048, num_classes, bias=True)



        if freeze_backbone:                   # hai dòng này tạo ra feature extraction

            for p in self.resnet.parameters():

                p.requires_grad = False



    def forward(self, images):

        features = self.resnet(images).pooler_output    # [B, 2048, 1, 1]

        features = features.squeeze(-1).squeeze(-1)     # [B, 2048]

        return self.classifier(features)                # [B, 21]



# --- Chạy Bài 4* (Resize 224x224 bắt buộc, chạy cả 2 chế độ: Feature Extraction và Fine-tuning) ---

train_dl_224, val_dl_224, test_dl_224, _ = get_dataloaders(img_size=224, use_augment=True)



# Chế độ 1: Feature Extraction (đóng băng backbone, lr = 1e-3)

resnet50_fe = PretrainedResnet(num_classes=len(CLASSES), freeze_backbone=True)

f1_r50_fe = run_experiment(resnet50_fe, train_dl_224, val_dl_224, test_dl_224, CLASSES, lr=1e-3, title="Bài 4*: ResNet-50 (Feature Extraction)")



# Chế độ 2: Fine-tuning toàn mạng (mở khoá backbone, lr = 1e-4)

resnet50_ft = PretrainedResnet(num_classes=len(CLASSES), freeze_backbone=False)

f1_r50_ft = run_experiment(resnet50_ft, train_dl_224, val_dl_224, test_dl_224, CLASSES, lr=1e-4, title="Bài 4*: ResNet-50 (Fine-tuning)")



print(f"So sánh Bài 4* -> Macro-F1 Feature Extraction: {f1_r50_fe:.4f} | Fine-tuning: {f1_r50_ft:.4f}")